# Base network from ChEA databases (BooleaBayes)

Build a base transcription-factor network from curated ChEA (and related) target interactions — the original BooleaBayes approach — for systems without scATAC-seq data.

> **Starter notebook.** The cells below are a scaffold: real function calls with placeholder paths/arguments and `TODO` markers. Fill in your own data and run top-to-bottom. Anything marked `TODO` is a choice you need to make for your dataset.

## Setup and imports

In [ ]:
import os
import os.path as op
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import bobaT as bb

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
import warnings
warnings.filterwarnings('ignore')

# ChEA / enrichment helpers live in the enrichr module
from bobaT import enrichr

## Configure paths and transcription factors

In [ ]:
# Input data
DATA_DIR = './test_data'

# Output directories
OUTPUT_DIR = './output'
VAL_DIR = './output/validation'
ATTRACTOR_DIR = './output/attractors'
PERTURB_DIR = './output/perturbations'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# TODO: the transcription factors of interest for your system
tfs = ['ASCL1', 'NEUROD1', 'POU2F3', 'YAP1']  # example

## Query enrichment databases for each TF's targets

Submit your gene list and query per-gene targets from the enrichment databases.

In [ ]:
# Submit the gene list once, then query targets
user_list_id = enrichr.submit_gene_list(tfs, description='BoBa-T TF set')

# TODO: collect target interactions for your TFs (see enrichr.query_gene /
# enrichr.enrich for the per-library query helpers)
chea_hits = enrichr.query_gene(tfs[0])  # placeholder for one gene

## Restrict to ChEA-sourced interactions

In [ ]:
# Keep only ChEA-sourced edges among your factors
# `data` here is the table of interactions returned above
chea_edges = enrichr.process_chea_lists(chea_hits, G=None)  # TODO: pass your table

## Build the transcription-factor network

In [ ]:
import graph_tool.all as gt
G = gt.Graph()
# Add edges from each TF to its ChEA targets that are also in `tfs`
for tf in tfs:
    G = enrichr.build_tf_network(G, tf, tfs)  # TODO: confirm per-TF loop for your data

## Prune the network

In [ ]:
# Drop weak edges, then restrict to ChEA-supported edges
G = enrichr.prune_weak_edges(G)
G = bb.net.prune_to_chea(G, prune_self_loops=True)

## Save the base network

In [ ]:
bb.net.save_network(f'{OUTPUT_DIR}/chea_network.csv', G, attributes=True, overwrite=True)
print('Saved base network to', f'{OUTPUT_DIR}/chea_network.csv')

## Next step

Use this network for [rule inference](inference_two_timepoints.ipynb).